# Task 1A: Naive RAG Pipeline

Basic RAG pipeline: fixed chunking (1024 tokens, overlap 200), dense retrieval (cosine), GPT-4o-mini generation.

**PDFs:** ktj.pdf (KTZh annual report), matnp_2024_rus.pdf (Maten Petroleum annual report)

In [ ]:
import sys
sys.path.insert(0, "..")

import json
from src.parsing import parse_all_pdfs
from src.pipeline import RAGPipeline
from src.config import DEFAULT_CONFIG

## 1. Parse PDFs
Uses LlamaParse with markdown output. Results are cached to `data/parsed/`.

In [ ]:
parsed_texts = parse_all_pdfs(use_cache=True)
for name, text in parsed_texts.items():
    print(f"{name}: {len(text)} chars")

## 2. Chunk & Index
Fixed chunking with 1024 tokens, 200 overlap. Embeddings: `intfloat/multilingual-e5-large`.

In [ ]:
# Naive RAG config: fixed chunking, dense retrieval only
naive_config = {
    **DEFAULT_CONFIG,
    "chunking_strategy": "fixed",
    "chunk_size": 1024,
    "chunk_overlap": 200,
    "alpha": 1.0,  # dense only
    "use_reranking": False,
    "use_query_rewriting": False,
    "collection_name": "naive_rag",
}

pipeline = RAGPipeline(naive_config)
n_chunks = pipeline.ingest(parsed_texts)
print(f"Indexed {n_chunks} chunks")

## 3. Demo: Single Query

In [ ]:
result = pipeline.query("Каков был доход от основной деятельности АО «НК «КТЖ» в 2024 году?")
print("Answer:", result["answer"])
print("\n--- Retrieved chunks ---")
for i, ctx in enumerate(result["contexts"], 1):
    print(f"\nChunk {i} ({len(ctx)} chars):")
    print(ctx[:300], "...")

## 4. Test on Golden Dataset (sample)

In [ ]:
with open("../data/golden_dataset.json", "r", encoding="utf-8") as f:
    golden = json.load(f)

# Test on first 10 questions
for item in golden[:10]:
    result = pipeline.query(item["question"])
    print(f"Q: {item['question']}")
    print(f"Expected: {item['ground_truth']}")
    print(f"Got: {result['answer']}")
    print("-" * 80)

## 5. Observed Problems

Common issues with Naive RAG:
1. **Tables split across chunks** — fixed chunking breaks table rows, losing context
2. **Exact names/numbers missed** — dense retrieval struggles with specific values (e.g., "1 875,6 млрд тенге")
3. **No keyword matching** — cosine similarity may miss lexically important terms
4. **Irrelevant chunks retrieved** — without reranking, top-K may include semantically similar but wrong passages

These problems motivate the Advanced RAG pipeline in Task 1B.